# Training and validation loss

Retrains the bass note model from scratch at the shipped hyperparameters to recover the
per-epoch loss history the original run never logged, and writes
`figures/loss-curves-bass.png`. Saves no checkpoint.

In [ ]:
import sys, pathlib, json

ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "experiments" / "loss-curves"))

import numpy as np
import matplotlib.pyplot as plt
from train_with_history import train_with_history

%config InlineBackend.figure_formats = ['retina']

In [ ]:
HIST = ROOT / "experiments" / "loss-curves" / "history_bass_augmented.json"

if HIST.exists():
    hist = json.load(open(HIST))
    print(f"loaded existing run: {hist['epochs']} epochs, {hist['seconds']}s")
else:
    hist = train_with_history(role="bass", strategy="augmented", epochs=150)
    HIST.parent.mkdir(parents=True, exist_ok=True)
    json.dump(hist, open(HIST, "w"), indent=1)

print(f"best val {min(hist['val']):.4f} at epoch {int(np.argmin(hist['val'])) + 1}")
print(f"{hist['n_train']:,} train / {hist['n_val']:,} val rows")
print(f"{hist['shared_phrases']} of {hist['n_train_phrases']} phrases appear on both sides")

In [ ]:
TRAIN, VAL = "#26699F", "#C2542B"
INK, MUTED, FAINT, RULE = "#16201F", "#63706E", "#8A9694", "#D5DCDA"


def _style(ax):
    ax.grid(color=RULE, lw=1)
    ax.set_axisbelow(True)
    for s in ("top", "right", "bottom", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0, colors=MUTED, labelsize=8.5)


def plot_loss_curves(hist, zoom_from=20, figsize=(11.6, 4.5)):
    tr, va = np.array(hist["train"]), np.array(hist["val"])
    ep = np.arange(1, len(tr) + 1)
    best = int(np.argmin(va))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    ax1.plot(ep, tr, color=TRAIN, lw=2, label="training", solid_capstyle="round")
    ax1.plot(ep, va, color=VAL, lw=2, label="validation", solid_capstyle="round")
    ax1.axvspan(zoom_from, len(tr), color=INK, alpha=.05, lw=0)
    ax1.annotate("zoomed →", (zoom_from + 2, tr.max() * .82), fontsize=8.5,
                 color=FAINT, family="monospace")
    ax1.set_xlim(0, len(tr) + 1)
    ax1.set_title("the whole run", fontsize=11, fontweight="bold", color=INK, loc="left", pad=10)
    ax1.set_ylabel("class-weighted cross-entropy — lower is better",
                   fontsize=8.5, color=MUTED, family="monospace", labelpad=9)
    ax1.legend(frameon=False, fontsize=9.5, labelcolor=MUTED, handlelength=1.6)
    _style(ax1)

    m = ep >= zoom_from
    ax2.fill_between(ep[m], tr[m], va[m], color=VAL, alpha=.13, lw=0)
    ax2.plot(ep[m], tr[m], color=TRAIN, lw=2, solid_capstyle="round")
    ax2.plot(ep[m], va[m], color=VAL, lw=2, solid_capstyle="round")
    ax2.scatter([ep[best]], [va[best]], s=54, color=VAL, zorder=5,
                edgecolor="white", linewidth=1.8)
    ax2.annotate(f"best val {va[best]:.4f}\nepoch {ep[best]}", (ep[best], va[best]),
                 textcoords="offset points", xytext=(-16, -34), ha="right", va="top",
                 fontsize=8.5, color=MUTED,
                 arrowprops=dict(arrowstyle="-", color=RULE, lw=1))
    mid = int(len(ep) * 0.45)
    gap = va[best] - tr[best]
    ax2.annotate(f"generalisation gap {gap:+.3f}", (ep[mid], (tr[mid] + va[mid]) / 2),
                 ha="center", va="center", fontsize=8.5, color=VAL, family="monospace",
                 bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=.75))
    ax2.set_xlim(zoom_from, len(tr) + 1)
    ax2.set_title(f"from epoch {zoom_from}", fontsize=11, fontweight="bold",
                  color=INK, loc="left", pad=10)
    _style(ax2)

    for ax in (ax1, ax2):
        ax.set_xlabel("epoch", fontsize=8.5, color=MUTED, family="monospace", labelpad=7)

    split = ("phrases held out — no transposed siblings shared"
             if hist.get("strategy") == "grouped"
             else f"random over the augmented corpus — "
                  f"{hist.get('shared_phrases','?')} of "
                  f"{hist.get('n_train_phrases','?')} phrases on both sides")
    fig.suptitle(f"{hist['role']}  ·  {hist['n_train']:,} train / {hist['n_val']:,} val rows"
                 f"  ·  split: {split}",
                 fontsize=9.5, color=MUTED, family="monospace", y=1.02, x=.02, ha="left")
    fig.subplots_adjust(wspace=.16)
    return fig


def gap_summary(hist):
    tr, va = np.array(hist["train"]), np.array(hist["val"])
    b = int(np.argmin(va))
    return {"best_val": round(float(va[b]), 4), "best_epoch": b + 1,
            "train_at_best": round(float(tr[b]), 4),
            "gap_at_best": round(float(va[b] - tr[b]), 4),
            "final_train": round(float(tr[-1]), 4), "final_val": round(float(va[-1]), 4),
            "val_rise_after_best": round(float(va[-1] - va[b]), 4)}

In [ ]:
fig = plot_loss_curves(hist, zoom_from=20)
fig.savefig("figures/loss-curves-bass.png", dpi=170,
            bbox_inches="tight", facecolor="white")
print(json.dumps(gap_summary(hist), indent=1))
fig;